# Smart Microscopy v2

**Config** — imports and all parameters.  
**Connect** — connect to LAS X.  
**Step 1** — set stage limits, strip template, enforce z-galvo.  
**Step 2** — draw the scan field in Navigator Expert, then run: reads and visualises tile positions.  
**Step 3** — set up the focus map: place focus markers on the scan field, run autofocus at each, fit Z plane.  
**Step 4** — acquire all tiles with interpolated Z.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path("controller/vendor/leica").resolve()))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import lasx
from lasx.scanning_templates import TEMPLATE_BASE, TEMPLATE_XML, STRIPPED_XML
from LasxApi import PYLICamApiConnector as lasxApi

# ── Stage limits (um) ────────────────────────────────────────────────
# Primary: place point markers at the sample boundary in Navigator Expert
# before running Step 1 — limits are derived from those positions.
# Fallback: set these values explicitly (leave as None to skip).
STAGE_X_MIN_UM   = None
STAGE_X_MAX_UM   = None
STAGE_Y_MIN_UM   = None
STAGE_Y_MAX_UM   = None
LIMIT_MARGIN_UM  = 500    # margin added around boundary markers or config limits
Z_GALVO_MIN      = -200
Z_GALVO_MAX      =  200
Z_WIDE_MIN       = -5000
Z_WIDE_MAX       =  5000

# ── Jobs ─────────────────────────────────────────────────────────────
AF_JOB           = "AF Job"
ACQUISITION_JOB  = "Overview"

# ── Autofocus ────────────────────────────────────────────────────────
RESTORE_AFTER_AF = True

In [ ]:
client = lasxApi.LasxApiClientPyModel
client.Connect("PythonClient")
client.PyApiClient.DelayInMilliseconds = 300
mode = client.PyApiSetApiInterfaceToUse.Model.ApiInterfaceToUse
client.PyApiSetApiInterfaceToUse.Model.ApiInterfaceToUse = (
    type(mode).Only_the_CAM_interface_is_used
)
assert lasx.ping(client), "LAS X not responding"

templates_dir = lasx.find_scanning_templates_dir()
print(f"Connected — templates: {templates_dir}")

## Step 1 — Sample boundary & template

Optionally place **point markers** at the corners of your sample in Navigator Expert — limits are derived from those positions.  
Fallback: set `STAGE_X/Y_MIN/MAX_UM` in the config cell.  
If neither is set, limits are derived from the scan field in Step 2 (no boundary shown).  
Strips the template and enforces z-galvo on all jobs.

In [ ]:
# ── Read boundary markers from Navigator Expert ───────────────────────
lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
_s1_data = lasx.parse_template_positions(templates_dir, TEMPLATE_BASE, client=client)

boundary_points = [
    g["center_um"]
    for g in _s1_data.get("geometries", {}).values()
    if g["type"] == "Point" and "center_um" in g
]

xy_config = [STAGE_X_MIN_UM, STAGE_X_MAX_UM, STAGE_Y_MIN_UM, STAGE_Y_MAX_UM]

if boundary_points:
    xs = [p["x_um"] for p in boundary_points]
    ys = [p["y_um"] for p in boundary_points]
    lasx.set_stage_limits(
        x_min=min(xs) - LIMIT_MARGIN_UM, x_max=max(xs) + LIMIT_MARGIN_UM,
        y_min=min(ys) - LIMIT_MARGIN_UM, y_max=max(ys) + LIMIT_MARGIN_UM,
        z_galvo_min=Z_GALVO_MIN, z_galvo_max=Z_GALVO_MAX,
        z_wide_min=Z_WIDE_MIN, z_wide_max=Z_WIDE_MAX,
    )
    _boundary_lim = lasx.get_stage_limits()
    print(f"Stage limits from {len(boundary_points)} boundary marker(s):")
elif all(v is not None for v in xy_config):
    lasx.set_stage_limits(
        x_min=STAGE_X_MIN_UM, x_max=STAGE_X_MAX_UM,
        y_min=STAGE_Y_MIN_UM, y_max=STAGE_Y_MAX_UM,
        z_galvo_min=Z_GALVO_MIN, z_galvo_max=Z_GALVO_MAX,
        z_wide_min=Z_WIDE_MIN, z_wide_max=Z_WIDE_MAX,
    )
    _boundary_lim = lasx.get_stage_limits()
    print(f"Stage limits from config:")
else:
    _boundary_lim = None
    print("No boundary markers or config limits — limits will be derived from scan field in Step 2.")

if _boundary_lim:
    print(f"  X: {_boundary_lim['x_min']:.0f} \u2013 {_boundary_lim['x_max']:.0f} um")
    print(f"  Y: {_boundary_lim['y_min']:.0f} \u2013 {_boundary_lim['y_max']:.0f} um")

# ── Strip template ────────────────────────────────────────────────────
strip_result = lasx.strip_template(client)
assert strip_result, "Strip failed"
print(f"\nTemplate stripped — draw scan field in Navigator Expert before running Step 2.")

# ── Enforce z-galvo ───────────────────────────────────────────────────
def _set_z_galvo(lrp_path):
    parsed = lasx.parse_lrp(lrp_path)
    for name in parsed["jobs"]:
        lasx.lrp_set_z_use_mode(lrp_path, "z-galvo", name)

def _verify_z_galvo(lrp_path):
    parsed = lasx.parse_lrp(lrp_path)
    return all(
        lasx.lrp_verify_z_use_mode(lrp_path, "z-galvo", n)
        for n in parsed["jobs"]
    )

result = lasx.apply_lrp_change(client, STRIPPED_XML, _set_z_galvo, verify_fn=_verify_z_galvo)
assert result and result["success"], "z-galvo enforcement failed"
print("z-galvo enforced on all jobs")

## Step 2 — Scan field

Draw the scanning field in **Navigator Expert**, then run this cell.

In [ ]:
lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
template_data = lasx.parse_template_positions(templates_dir, TEMPLATE_BASE, client=client)

tile_positions = template_data.get("acquisition_positions", {})
if not tile_positions and template_data.get("geometries"):
    from lasx.scanning_template_parsers import _tile_size_from_image_size_str
    settings = lasx.get_job_settings(client, ACQUISITION_JOB)
    tile_size_um = None
    if settings:
        tile_size_um = _tile_size_from_image_size_str(settings.get("imageSize", ""))
    if tile_size_um:
        template_data = lasx.synthesize_tiles(template_data, tile_size_um, job_name=ACQUISITION_JOB)
        tile_positions = template_data["acquisition_positions"]
        n_synth = sum(len(r["positions"]) for r in tile_positions.values())
        print(f"Synthesized {n_synth} tiles from geometries (tile size {tile_size_um:.1f} um)")
    else:
        print("WARNING: Cannot determine tile size — no tiles generated")

n_tiles = sum(len(r["positions"]) for r in tile_positions.values())
print(f"Scan field: {len(tile_positions)} region(s), {n_tiles} tile(s)")
for rid, region in tile_positions.items():
    print(f"  Region {rid}: {region['job_name']}  "
          f"{region.get('num_rows', '?')}\u00d7{region.get('num_cols', '?')}  "
          f"tile={region.get('tile_size_um', '?')} um")

# ── Stage limits ─────────────────────────────────────────────────────
lim = globals().get("_boundary_lim")

if lim:
    print(f"\nStage limits: X {lim['x_min']:.0f} \u2013 {lim['x_max']:.0f}  "
          f"Y {lim['y_min']:.0f} \u2013 {lim['y_max']:.0f} um")
else:
    tile_centers_x = [p["x_um"] for r in tile_positions.values() for p in r["positions"]]
    tile_centers_y = [p["y_um"] for r in tile_positions.values() for p in r["positions"]]
    if tile_centers_x:
        ts_half = max((r.get("tile_size_um") or 0) for r in tile_positions.values()) / 2
        lasx.set_stage_limits(
            x_min=min(tile_centers_x) - ts_half - LIMIT_MARGIN_UM,
            x_max=max(tile_centers_x) + ts_half + LIMIT_MARGIN_UM,
            y_min=min(tile_centers_y) - ts_half - LIMIT_MARGIN_UM,
            y_max=max(tile_centers_y) + ts_half + LIMIT_MARGIN_UM,
            z_galvo_min=Z_GALVO_MIN, z_galvo_max=Z_GALVO_MAX,
            z_wide_min=Z_WIDE_MIN, z_wide_max=Z_WIDE_MAX,
        )
        print("\nNo sample boundary defined — stage limits derived from scan field for safety.")

# ── Visualise ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f5f5f8")

all_view_x, all_view_y = [], []

viz_colors = template_data.get("visualization_data", {}).get("tile_colors", {})
job_color_map = {
    region["job_name"]: tuple(viz_colors[region["job_name"]])
    for region in tile_positions.values()
    if region["job_name"] in viz_colors
}
legend_jobs = set()

for rid, region in tile_positions.items():
    jn = region["job_name"]
    ts = region.get("tile_size_um")
    if ts is None:
        continue
    half = ts / 2
    rgba = job_color_map.get(jn, (0.78, 0.78, 0.78, 1.0))
    face = (rgba[0], rgba[1], rgba[2], 0.25)
    edge = (rgba[0], rgba[1], rgba[2], 0.80)
    for pos in region["positions"]:
        cx, cy = pos["x_um"], pos["y_um"]
        ax.add_patch(patches.Rectangle(
            (cx - half, cy - half), ts, ts,
            linewidth=0.6, edgecolor=edge, facecolor=face, zorder=2,
        ))
        all_view_x.extend([cx - half, cx + half])
        all_view_y.extend([cy - half, cy + half])
    if jn not in legend_jobs:
        label = "No job assigned" if jn == "(unassigned)" else jn
        ax.plot([], [], "s", color=(rgba[0], rgba[1], rgba[2], 0.6), markersize=8, label=label)
        legend_jobs.add(jn)

if lim:
    ax.add_patch(patches.Rectangle(
        (lim["x_min"], lim["y_min"]),
        lim["x_max"] - lim["x_min"],
        lim["y_max"] - lim["y_min"],
        linewidth=1.0, edgecolor="#aaaaaa", facecolor="none",
        linestyle=(0, (5, 4)), zorder=1,
    ))
    ax.plot([], [], ls=(0, (5, 4)), color="#aaaaaa", linewidth=1.0, label="Sample boundary")
    all_view_x.extend([lim["x_min"], lim["x_max"]])
    all_view_y.extend([lim["y_min"], lim["y_max"]])

if all_view_x:
    span = max(max(all_view_x) - min(all_view_x), max(all_view_y) - min(all_view_y))
    cross = span * 0.006
    circle_r = cross * 0.6
    for fp_list, color, label in [
        (template_data.get("focus_points", []),     "#4EB8B8", "Focus points"),
        (template_data.get("autofocus_points", []), "#5CBF5C", "AutoFocus points"),
    ]:
        for fp in fp_list:
            fx, fy = fp["x_um"], fp["y_um"]
            ax.plot([fx - cross, fx + cross], [fy, fy], "-", color=color, linewidth=0.8, zorder=10)
            ax.plot([fx, fx], [fy - cross, fy + cross], "-", color=color, linewidth=0.8, zorder=10)
            ax.add_patch(patches.Circle((fx, fy), circle_r,
                linewidth=0.8, edgecolor=color, facecolor="none", zorder=11))
            all_view_x.append(fx); all_view_y.append(fy)
        if fp_list:
            ax.plot([], [], "+", color=color, markersize=10, markeredgewidth=1.8, label=label)

    pad = span * 0.05
    ax.set_xlim(min(all_view_x) - pad, max(all_view_x) + pad)
    ax.set_ylim(min(all_view_y) - pad, max(all_view_y) + pad)

ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xticks([]); ax.set_yticks([])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(0.8); spine.set_edgecolor("#cccccc")
ax.set_title("Scan Field", fontsize=13, fontweight="bold", color="#222222", pad=12)
ax.legend(loc="upper right", fontsize=9, facecolor="white", edgecolor="#cccccc", labelcolor="#444444")
plt.tight_layout()
plt.show()

## Step 3 — Focus map

Place **focus markers** on the scan field in **Navigator Expert**, then run this cell.  
Runs the AF job at each marker, reads back Z, fits a plane.

In [ ]:
lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
_af_data = lasx.parse_template_positions(templates_dir, TEMPLATE_BASE, client=client)

focus_positions = (
    _af_data.get("focus_points", []) or
    _af_data.get("autofocus_points", [])
)
if not focus_positions:
    raise RuntimeError(
        "No focus points found.\n"
        "Add focus or autofocus positions on the scan field in Navigator Expert, "
        "then re-run this cell."
    )

print(f"Focus positions ({len(focus_positions)}):")
for i, fp in enumerate(focus_positions):
    print(f"  {i + 1}. x={fp['x_um']:.1f}  y={fp['y_um']:.1f} um")

lasx.strip_template(client)
lasx.select_job(client, AF_JOB)

measured_z = []
for i, fp in enumerate(focus_positions):
    print(f"\n[{i + 1}/{len(focus_positions)}] x={fp['x_um']:.0f}  y={fp['y_um']:.0f}",
          end="", flush=True)
    lasx.move_xy(client, fp["x_um"], fp["y_um"])
    lasx.acquire(client, AF_JOB)
    settings = lasx.get_job_settings(client, AF_JOB)
    ch = lasx.make_changeable_copy(settings)
    z_um = ch["zPosition"]["z-galvo"]
    measured_z.append({**fp, "z_um": z_um})
    print(f"  z={z_um:.2f} um")

if RESTORE_AFTER_AF:
    lasx.restore_template(client)

xs = np.array([m["x_um"] for m in measured_z])
ys = np.array([m["y_um"] for m in measured_z])
zs = np.array([m["z_um"] for m in measured_z])
A = np.column_stack([xs, ys, np.ones(len(measured_z))])
plane_coeffs, *_ = np.linalg.lstsq(A, zs, rcond=None)

def interpolate_z(x, y):
    return plane_coeffs[0] * x + plane_coeffs[1] * y + plane_coeffs[2]

residuals = zs - np.array([interpolate_z(x, y) for x, y in zip(xs, ys)])
print(f"\nFocus plane:")
print(f"  Z range:      {zs.max() - zs.min():.2f} um")
print(f"  Tilt X:       {np.degrees(np.arctan(plane_coeffs[0])):+.4f} deg")
print(f"  Tilt Y:       {np.degrees(np.arctan(plane_coeffs[1])):+.4f} deg")
print(f"  Max residual: {np.max(np.abs(residuals)):.3f} um")

# ── Heatmap ──────────────────────────────────────────────────────────
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1 import make_axes_locatable

tile_data = [
    (p["x_um"], p["y_um"], r.get("tile_size_um", 0) / 2)
    for r in tile_positions.values()
    for p in r["positions"]
]

lim = globals().get("_boundary_lim")

fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f5f5f8")

if tile_data:
    tcx = [t[0] for t in tile_data]
    tcy = [t[1] for t in tile_data]
    th  = [t[2] for t in tile_data]

    all_tx = [cx - h for cx, h in zip(tcx, th)] + [cx + h for cx, h in zip(tcx, th)]
    all_ty = [cy - h for cy, h in zip(tcy, th)] + [cy + h for cy, h in zip(tcy, th)]
    span = max(max(all_tx) - min(all_tx), max(all_ty) - min(all_ty))

    step = span / 500 if span > 0 else 1.0
    pad  = 3 * step
    gx = np.arange(min(all_tx) - pad, max(all_tx) + pad + step, step)
    gy = np.arange(min(all_ty) - pad, max(all_ty) + pad + step, step)
    GX, GY = np.meshgrid(gx, gy)
    GZ = interpolate_z(GX, GY)

    norm = Normalize(vmin=GZ.min(), vmax=GZ.max())
    cmap = plt.get_cmap("viridis")

    verts, codes = [], []
    for cx, cy, h in zip(tcx, tcy, th):
        x0, y0, x1, y1 = cx - h, cy - h, cx + h, cy + h
        verts += [(x0, y0), (x1, y0), (x1, y1), (x0, y1), (x0, y0)]
        codes += [MplPath.MOVETO, MplPath.LINETO, MplPath.LINETO,
                  MplPath.LINETO, MplPath.CLOSEPOLY]
    clip_patch = PathPatch(MplPath(verts, codes),
                           transform=ax.transData, facecolor="none", edgecolor="none")
    ax.add_patch(clip_patch)

    im = ax.imshow(
        GZ, cmap=cmap, norm=norm, origin="lower", aspect="equal",
        extent=[gx[0], gx[-1] + step, gy[0], gy[-1] + step],
        interpolation="bilinear", zorder=1,
    )
    im.set_clip_path(clip_patch)

    for cx, cy, h in zip(tcx, tcy, th):
        ax.add_patch(patches.Rectangle(
            (cx - h, cy - h), 2 * h, 2 * h,
            linewidth=0.5, edgecolor="white", facecolor="none", zorder=3))

    cross = span * 0.006
    for m in measured_z:
        fx, fy = m["x_um"], m["y_um"]
        ax.plot([fx - cross, fx + cross], [fy, fy], "-", color="#e05555", linewidth=0.8, zorder=10)
        ax.plot([fx, fx], [fy - cross, fy + cross], "-", color="#e05555", linewidth=0.8, zorder=10)
        ax.add_patch(patches.Circle((fx, fy), cross * 0.6,
            linewidth=0.8, edgecolor="#e05555", facecolor="none", zorder=11))
    ax.plot([], [], "+", color="#e05555", markersize=10, markeredgewidth=1.8, label="Focus points")

    all_view_x = list(all_tx)
    all_view_y = list(all_ty)
    if lim:
        ax.add_patch(patches.Rectangle(
            (lim["x_min"], lim["y_min"]),
            lim["x_max"] - lim["x_min"],
            lim["y_max"] - lim["y_min"],
            linewidth=1.0, edgecolor="#aaaaaa", facecolor="none",
            linestyle=(0, (5, 4)), zorder=2,
        ))
        ax.plot([], [], ls=(0, (5, 4)), color="#aaaaaa", linewidth=1.0, label="Sample boundary")
        all_view_x.extend([lim["x_min"], lim["x_max"]])
        all_view_y.extend([lim["y_min"], lim["y_max"]])

    view_span = max(max(all_view_x) - min(all_view_x), max(all_view_y) - min(all_view_y))
    pad_plot = view_span * 0.05
    ax.set_xlim(min(all_view_x) - pad_plot, max(all_view_x) + pad_plot)
    ax.set_ylim(min(all_view_y) - pad_plot, max(all_view_y) + pad_plot)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="2%", pad=0.15)
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, cax=cax, label="Z (um)")

ax.set_aspect("equal"); ax.invert_yaxis()
ax.set_xticks([]); ax.set_yticks([])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(0.8); spine.set_edgecolor("#cccccc")
z_range = zs.max() - zs.min()
ax.set_title(f"Focus Plane  (range {z_range:.2f} um)",
             fontsize=13, fontweight="bold", color="#222222", pad=12)
ax.legend(loc="upper right", fontsize=9, facecolor="white", edgecolor="#cccccc", labelcolor="#444444")
plt.tight_layout()
plt.show()

## Step 4 — Acquire

Strips the template, acquires at every tile position with interpolated Z from the focus plane.

In [ ]:
lasx.strip_template(client)
lasx.select_job(client, ACQUISITION_JOB)

sequence = []
for rid, region in sorted(tile_positions.items(), key=lambda r: int(r[0])):
    rows = {}
    for p in region["positions"]:
        rows.setdefault(p["row"], []).append(p)
    for row_idx in sorted(rows):
        row_tiles = sorted(rows[row_idx], key=lambda p: p["col"])
        if row_idx % 2 == 1:
            row_tiles = row_tiles[::-1]
        for p in row_tiles:
            sequence.append({
                "region": rid,
                "x_um": p["x_um"],
                "y_um": p["y_um"],
                "z_um": interpolate_z(p["x_um"], p["y_um"]),
            })

print(f"Acquiring {len(sequence)} positions with '{ACQUISITION_JOB}'\n")

results = []
for i, pos in enumerate(sequence):
    print(f"[{i + 1}/{len(sequence)}] R{pos['region']}  "
          f"x={pos['x_um']:.0f}  y={pos['y_um']:.0f}  "
          f"z={pos['z_um']:.2f}", end="", flush=True)
    lasx.move_xy(client, pos["x_um"], pos["y_um"])
    lasx.move_z(client, ACQUISITION_JOB, pos["z_um"], z_mode="galvo")
    result = lasx.acquire(client, ACQUISITION_JOB)
    results.append({**pos, "success": result["success"]})
    elapsed = result["timing"]["total_s"] if result["success"] else 0
    status = "OK" if result["success"] else "FAIL"
    print(f"  {status} ({elapsed:.1f}s)")

ok = sum(1 for r in results if r["success"])
print(f"\nDone: {ok}/{len(results)} successful")